# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
## Explore available record sets and their fields, referencing by `@id`
record_sets = metadata.recordSet
if record_sets is None or len(record_sets) == 0:
    print("No record sets found in dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        # List fields for each record set
        if 'field' in rs:
            fields = rs['field']
            for field in fields:
                print(f"  Field @id: {field['@id']} Name: {field.get('name', '')}")
        else:
            print("  No fields listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If record sets exist, extract their @id values for loading.
record_sets_ids = []
if record_sets is not None:
    record_sets_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

# Load each record set into a DataFrame
for rs_id in record_sets_ids:
    print(f"Loading records from RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Columns for RecordSet @id {rs_id}: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(), "\n")
    else:
        print(f"No records found for RecordSet @id {rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll select the first available record set for demonstration if any are present.
if len(dataframes) > 0:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Using RecordSet @id: {rs_id} for EDA")

    # Find numeric columns (fields) by simple dtype inference
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use the first numeric field
        threshold = df[numeric_field].mean()  # Use the mean as a threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a categorical/group field
        group_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field = group_candidates[0] if group_candidates else None
        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field (categorical) found for grouping.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No DataFrames found to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic distribution visualization
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_cols:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field} in RecordSet @id: {rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we've:

- Loaded and summarized the Kilifi County Mental Health Survey dataset using the Croissant schema and `mlcroissant`.
- Inspected metadata and listed available record sets and their fields, referenced by `@id`.
- Loaded records into DataFrames and performed typical exploratory data tasks, including filtering and normalization on numeric fields, and grouping by categorical attributes where possible.
- Visualized the distribution of key numeric indicators for mental health.

This approach allows flexible exploration of survey-based datasets packaged in Croissant, ensuring all references and analysis steps are traceable via schema `@id`s.